# Hosted banking-feedback application checks

Complete collection, source verification, responsive interfaces and fixed agent tasks.


## Source and task bundle


In [ ]:
import base64
import hashlib
import io
import json
import os
import subprocess
import sys
import time
import zipfile
from pathlib import Path
started=time.monotonic()
notebook=next(p for p in [Path('__notebook__.ipynb'),Path('/kaggle/working/__notebook__.ipynb')] if p.exists())
submitted=json.loads(notebook.read_text())
payload=next(c['attachments']['live_assets.zip']['application/zip'] for c in submitted['cells'] if 'live_assets.zip' in c.get('attachments',{}))
root=Path('/kaggle/working/live_assets')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(payload))) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist()), 'UNSAFE_ASSET_PATH'
    manifest=json.loads(z.read('checksums.json'))
    assert set(z.namelist())=={v['path'] for v in manifest}|{'checksums.json'}, 'ASSET_MEMBER_MISMATCH'
    for item in manifest:
        raw=z.read(item['path'])
        assert len(raw)==item['bytes'] and hashlib.sha256(raw).hexdigest()==item['sha256'], 'ASSET_HASH_MISMATCH'
    z.extractall(root)
os.environ['LIVE_ASSET_ROOT']=str(root)
sys.path.insert(0,str(root))
print('RESOLVED_ASSETS',str(root),len(manifest))
subprocess.check_call([sys.executable,'-m','pip','install','-q','playwright==1.55.0'])
subprocess.check_call([sys.executable,'-m','playwright','install','--with-deps','chromium'])


## Source checks


In [ ]:
from source_audit import source_audit
audits=[source_audit(p) for p in root.glob('*.py')]
directory=Path('/kaggle/working/hosted_source_checks')
directory.mkdir(exist_ok=True)
cell_audits=[]
for i,cell in enumerate(submitted['cells']):
    if cell['cell_type']=='code':
        p=directory/f'cell_{i:03d}.py'
        p.write_text(''.join(cell['source']))
        cell_audits.append(source_audit(p))
(directory/'source_audits.json').write_text(json.dumps(audits,indent=2))
(directory/'notebook_code_audit.json').write_text(json.dumps(cell_audits,indent=2))
print('SOURCE_AUDITS',json.dumps(audits))
print('NOTEBOOK_SOURCE_AUDIT',json.dumps(cell_audits))


## Hosted interface and model checks


In [ ]:
try:
    subprocess.run([sys.executable,str(root/'live_app_checks.py')],check=True)
finally:
    print('TOTAL_NOTEBOOK_SECONDS',round(time.monotonic()-started,3))


## Retained task results and artifact


In [ ]:
import shutil
output=Path('/kaggle/working/hosted_app_checks')
result=json.loads((output/'hosted_checks.json').read_text())
assert result['status']=='complete', 'INCOMPLETE_HOSTED_CHECKS'
review=output/'reviewer_only'
review.mkdir(exist_ok=True)
for name in ['source_audits.json','notebook_code_audit.json']:
    shutil.copy2(directory/name,review/name)
for name in ['hosted_source_hashes.json','live_app_cases.json']:
    shutil.copy2(root/name,review/name)
files=[p for p in output.rglob('*') if p.is_file() and p.name not in ['checksums.json','verification_full_export.csv','verification_decisions.json']]
inventory=[{'path':p.relative_to(output).as_posix(),'bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(files)]
(output/'checksums.json').write_text(json.dumps(inventory,indent=2))
archive=output.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in files+[output/'checksums.json']:
        z.write(p,p.relative_to(output).as_posix())
print('HOSTED_TASK_SUMMARY',json.dumps({'successes':result['task_successes'],'tasks':result['task_total'],'rows':result['corpus_rows'],'export_rows':result['full_export']['rows'],'interface_checks':len(result['interface_checks'])}))
print('ARTIFACT_ZIP',str(archive),archive.stat().st_size,hashlib.sha256(archive.read_bytes()).hexdigest())
